# 01 - Data Preparation

Pulls (or reuses the on-disk cache of) all raw data sources, cleans the FRED-MD panel, engineers the full feature set, and builds both labels (recession + market-stress). This mirrors exactly what `src/pipeline.py` does -- this notebook calls the same functions rather than duplicating logic, so there is only one source of truth.

See `README.md` and the module docstrings in `src/data_ingestion.py` / `src/fredmd_spec.py` for the full honesty notes on data sourcing (why a handful of FRED-MD series are reconstructed or dropped).

In [1]:
import sys
sys.path.insert(0, '../src')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 4.5)

from pipeline import load_config, load_raw_inputs, clean_panel, build_features, build_label
from feature_engineering import align_features_to_horizon

cfg = load_config()
cfg['data'], cfg['cleaning']

({'vintage_pull_date': '2026-09-06',
  'fredmd_start_date': '1959-01-01',
  'raw_dir': 'data/raw',
  'processed_dir': 'data/processed',
  'datasets_dir': 'data/datasets',
  'predictions_dir': 'data/predictions'},
 {'max_null_rows': 6, 'max_null_cols': 10})

## 1. Raw data coverage

How many FRED-MD series were reconstructed live from FRED vs. dropped, and why.

In [2]:
ingestion_log = pd.read_csv('../data/raw/fredmd_ingestion_log.csv')
ingestion_log['status'].value_counts()

status
ok                                            101
DROPPED (short history in free substitute)     11
ok (reconstructed)                              8
DROPPED (no free source)                        5
ok (substitute source)                          1
Name: count, dtype: int64

In [3]:
ingestion_log[ingestion_log['status'].str.contains('DROPPED')][['fredmd_id','description']]

,fredmd_id,description
110,HWI,Help-Wanted Index discontinued by Conference B...
111,HWIURATIO,Depends on HWI (see above).
112,SP_INDUST,S&P's Common Stock Price Index: Industrials --...
113,SP_DIV_YIELD,S&P Composite dividend yield -- proprietary S&...
114,SP_PE_RATIO,S&P Composite P/E ratio -- same proprietary/st...
115,RETAILx,Substitute RSAFS only starts 1992-01 (FRED-MD'...
116,ACOGNO,FRED series ACOGNO itself only starts 1992-02.
117,AMDMNOx,Substitute DGORDER only starts 1992-02.
118,ANDENOx,Substitute ANDENO only starts 1992-02.
119,AMDMUOx,Substitute AMDMUO only starts 1992-01.


## 2. Load raw panel + clean

In [4]:
data, tcodes, usrec, vix_monthly, series_spec = load_raw_inputs(cfg)
print('Raw FRED-MD panel (reconstructed):', data.shape, data.index.min().date(), 'to', data.index.max().date())

cleaned, clean_log = clean_panel(data, cfg)
print(clean_log)
print('\nCleaned panel:', cleaned.shape, cleaned.index.min().date(), 'to', cleaned.index.max().date())

2026-09-06 07:45:57,068 INFO remove_null_rows(max_null=6): dropped 1071/1869 rows (date range kept: 1960-01-01 to 2026-07-01)


2026-09-06 07:45:57,070 INFO remove_null_features(max_null=10): dropped 7/110 columns: ['CLAIMSx', 'CMRMTSPLx', 'AMBSL', 'EXSZUSx', 'EXJPUSx', 'EXUSUKx', 'EXCAUSx']


2026-09-06 07:45:57,072 INFO fill_null_obs: forward-filled 4 values (0 remain null, typically leading-edge NaNs before a series' inception)


Raw FRED-MD panel (reconstructed): (1869, 110) 1871-01-01 to 2026-09-01
remove_null_rows(max_null=6): dropped 1071/1869 rows (date range kept: 1960-01-01 to 2026-07-01)
remove_null_features(max_null=10): dropped 7/110 columns: ['CLAIMSx', 'CMRMTSPLx', 'AMBSL', 'EXSZUSx', 'EXJPUSx', 'EXUSUKx', 'EXCAUSx']
fill_null_obs: forward-filled 4 values (0 remain null, typically leading-edge NaNs before a series' inception)

Cleaned panel: (798, 103) 1960-01-01 to 2026-07-01


## 3. Feature engineering

Base (contemporaneous FRED-MD transform) + multi-horizon lags + spreads + momentum + rolling stats + drawdowns + volatility.

In [5]:
feat_result = build_features(cleaned, tcodes, series_spec, vix_monthly, cfg)
print('Total engineered features:', feat_result.features.shape[1])
feat_result.metadata['feat_type'].value_counts()

2026-09-06 07:45:57,451 INFO Built 1880 engineered features (base=103, lag=515, spread=5, momentum=309, rolling_stat=927, drawdown=12, volatility=9)


Total engineered features: 1880


feat_type
rolling_stat    927
lag             515
momentum        309
base            103
drawdown         12
volatility        9
spread            5
Name: count, dtype: int64

In [6]:
feat_result.metadata['group'].value_counts()

group
Labor Market                           453
Prices                                 414
Output and Income                      291
Interest and Exchange Rates            275
Money and Credit                       216
Housing                                183
Stock Market                            30
Consumption, Orders and Inventories     18
Name: count, dtype: int64

## 4. Labels

### Recession (NBER USREC)

In [7]:
recession_label = build_label('recession', cleaned, usrec, vix_monthly, cfg)
recession_label.value_counts(dropna=False)

label
0    703
1     95
Name: count, dtype: int64

### Market stress (S&P 500 drawdown AND elevated VIX)

In [8]:
market_label = build_label('market_stress', cleaned, usrec, vix_monthly, cfg)
market_label.value_counts(dropna=False)

2026-09-06 07:45:57,484 INFO Market-stress label: 403/798 months labeled (rest undefined pre-warm-up), 7.2% positive (stress) among labeled


label
NaN    395
0.0    374
1.0     29
Name: count, dtype: int64

In [9]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(cleaned.index, cleaned['SP500'], color='black', linewidth=0.9)
axes[0].set_yscale('log')
axes[0].set_ylabel('S&P 500 (log scale)')
for d, v in recession_label.items():
    pass
rec = recession_label.reindex(cleaned.index).fillna(0)
in_reg, start = False, None
for d, v in rec.items():
    if v == 1 and not in_reg:
        in_reg, start = True, d
    elif v == 0 and in_reg:
        axes[0].axvspan(start, d, color='grey', alpha=0.3)
        in_reg = False
axes[0].set_title('S&P 500 with NBER recessions shaded')

axes[1].plot(vix_monthly.index, vix_monthly['VIX_MEAN'], color='darkred', linewidth=0.9)
axes[1].set_ylabel('VIX (monthly avg)')
mkt = market_label.reindex(cleaned.index).fillna(0)
in_reg, start = False, None
for d, v in mkt.items():
    if v == 1 and not in_reg:
        in_reg, start = True, d
    elif v == 0 and in_reg:
        axes[1].axvspan(start, d, color='grey', alpha=0.3)
        in_reg = False
axes[1].set_title('VIX with market-stress regime shaded')
plt.tight_layout()
plt.savefig('../results/figures/01_regimes_overview.png', dpi=130)
plt.show()

## 5. Horizon alignment sanity check (no lookahead)

Features are shifted FORWARD by the 1-month horizon (not the label shifted backward) -- see `feature_engineering.align_features_to_horizon`.

In [10]:
h = cfg['split']['horizon']
Xa = align_features_to_horizon(feat_result.features, h)
# spot check: row t's value should equal the ORIGINAL feature at row t-h
raw = feat_result.features['SP500__base']
shifted = Xa['SP500__base']
check = (shifted.iloc[h+5] == raw.iloc[5])
print('Alignment self-check (row h+5 of shifted == row 5 of raw):', check)

Alignment self-check (row h+5 of shifted == row 5 of raw): True


Both datasets (features + label, aligned and NaN-dropped) are built and persisted by `src/pipeline.py::run_target`, which is what `02_recession_model.ipynb` and `03_market_stress_model.ipynb` call next.